In [1]:
%pip install numpy pandas scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, r2_score
from sklearn.preprocessing import label_binarize
import time

class KNNClassifier:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train

    def predict(self, X_test):
        predictions = []
        for x in X_test:
            distances = np.linalg.norm(self.X_train - x, axis=1)  
            nearest_indices = distances.argsort()[:self.k]
            nearest_labels = self.y_train[nearest_indices]
            
            class_votes = np.zeros(len(np.unique(self.y_train)))
            for label in nearest_labels:
                class_votes[label] += 1
            
            predictions.append(np.argmax(class_votes))
        return np.array(predictions)


data_file = "../data/data.csv"  
import pandas as pd
df = pd.read_csv(data_file)

X = df.drop(columns=["Model", "Engine Capacity", "Horsepower"]).values
y = df["Model"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
k_values = range(1, 25)
cv_scores = []

for k in k_values:
    fold_accuracies = []
    for train_index, val_index in kf.split(X_train):
        X_fold_train, X_fold_val = X_train[train_index], X_train[val_index]
        y_fold_train, y_fold_val = y_train[train_index], y_train[val_index]

        knn = KNNClassifier(k=k)
        knn.fit(X_fold_train, y_fold_train)
        y_val_pred = knn.predict(X_fold_val)
        fold_accuracies.append(accuracy_score(y_fold_val, y_val_pred))
    
    cv_scores.append(np.mean(fold_accuracies))

best_k = k_values[np.argmax(cv_scores)]
print(f"Best k value: {best_k}, Cross-Validation Accuracy: {max(cv_scores)}")

start_time = time.time()
final_knn = KNNClassifier(k=best_k)
final_knn.fit(X_train, y_train)

y_train_pred = final_knn.predict(X_train)
y_test_pred = final_knn.predict(X_test)
end_time = time.time()

def evaluate_model(y_true, y_pred, dataset_name="Dataset"):
    print(f"\n--- {dataset_name} Performance Metrics ---")
    print(f"Accuracy: {accuracy_score(y_true, y_pred)}")
    print(f"F1 Score: {f1_score(y_true, y_pred, average='weighted')}")
    print(f"Precision: {precision_score(y_true, y_pred, average='weighted')}")
    print(f"Recall: {recall_score(y_true, y_pred, average='weighted')}")

evaluate_model(y_train, y_train_pred, dataset_name="Training Set")
evaluate_model(y_test, y_test_pred, dataset_name="Test Set")
print(f"Model Training and Execution Time: {end_time - start_time} seconds")

def calculate_auroc_one_vs_all(y_true, y_pred_proba):
    y_bin = label_binarize(y_true, classes=np.unique(y_true))  
    return roc_auc_score(y_bin, y_pred_proba, multi_class='ovr')  


y_train_pred_proba = np.zeros((len(y_train), len(np.unique(y_train))))
y_test_pred_proba = np.zeros((len(y_test), len(np.unique(y_train))))

for i, model in enumerate(np.unique(y_train)):
    y_train_pred_proba[:, i] = (y_train_pred == model).astype(int)
    y_test_pred_proba[:, i] = (y_test_pred == model).astype(int)

train_auroc = calculate_auroc_one_vs_all(y_train, y_train_pred_proba)
test_auroc = calculate_auroc_one_vs_all(y_test, y_test_pred_proba)

print(f"\nTraining Set AUROC (One-vs-All): {train_auroc}")
print(f"Test Set AUROC (One-vs-All): {test_auroc}")

r2_score = r2_score(y_test, y_test_pred)
print(f"R2 Score: {r2_score}")




Best k value: 12, Cross-Validation Accuracy: 0.512216941571149

--- Training Set Performance Metrics ---
Accuracy: 0.603542234332425
F1 Score: 0.5976673778953631
Precision: 0.597959595993419
Recall: 0.603542234332425

--- Test Set Performance Metrics ---
Accuracy: 0.5163043478260869
F1 Score: 0.5130814620091352
Precision: 0.5151357474901364
Recall: 0.5163043478260869
Model Training and Execution Time: 0.04401421546936035 seconds

Training Set AUROC (One-vs-All): 0.7021200928818353
Test Set AUROC (One-vs-All): 0.6338652006995934
R2 Score: -0.549509531741625


In [11]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, r2_score
from sklearn.preprocessing import label_binarize
import pandas as pd
import time


data_file = "../data/data.csv"  
df = pd.read_csv(data_file)

X = df.drop(columns=["Model", "Engine Capacity", "Horsepower"]).values  
y = df["Model"].values 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

k_values = range(1, 21)
cv_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

best_k = k_values[cv_scores.index(max(cv_scores))]
print(f"Best k value: {best_k}, Cross-Validation Accuracy: {max(cv_scores)}")

start_time = time.time()
final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train, y_train)

y_train_pred = final_knn.predict(X_train)
y_test_pred = final_knn.predict(X_test)
end_time = time.time()

def evaluate_model(y_true, y_pred, dataset_name="Dataset"):
    print(f"\n--- {dataset_name} Performance Metrics ---")
    print(f"Accuracy: {accuracy_score(y_true, y_pred)}")
    print(f"F1 Score: {f1_score(y_true, y_pred, average='weighted')}")
    print(f"Precision: {precision_score(y_true, y_pred, average='weighted')}")
    print(f"Recall: {recall_score(y_true, y_pred, average='weighted')}")

evaluate_model(y_train, y_train_pred, dataset_name="Training Set")
evaluate_model(y_test, y_test_pred, dataset_name="Test Set")
print(f"Model Training and Execution Time: {end_time - start_time} seconds")

y_train_pred_proba = np.zeros((len(y_train), len(np.unique(y_train))))
y_test_pred_proba = np.zeros((len(y_test), len(np.unique(y_train))))

for i, model in enumerate(np.unique(y_train)):
    y_train_pred_proba[:, i] = (y_train_pred == model).astype(int)
    y_test_pred_proba[:, i] = (y_test_pred == model).astype(int)

train_auroc = calculate_auroc_one_vs_all(y_train, y_train_pred_proba)
test_auroc = calculate_auroc_one_vs_all(y_test, y_test_pred_proba)

print(f"\nTraining Set AUROC: {train_auroc}")
print(f"Test Set AUROC: {test_auroc}")

r2_score = r2_score(y_test, y_test_pred)
print(f"R2 Score: {r2_score}")


Best k value: 10, Cross-Validation Accuracy: 0.5123101295312645

--- Training Set Performance Metrics ---
Accuracy: 0.6239782016348774
F1 Score: 0.6191865493927792
Precision: 0.6189844154794492
Recall: 0.6239782016348774

--- Test Set Performance Metrics ---
Accuracy: 0.5217391304347826
F1 Score: 0.5209375631286662
Precision: 0.5243271221532091
Recall: 0.5217391304347826
Model Training and Execution Time: 0.017014265060424805 seconds

Training Set AUROC: 0.717445391633822
Test Set AUROC: 0.638498810903472
R2 Score: -0.566537109013511
